# Source Extractor Tutorial

In this notebook we will learn how to:
1. Run **Source Extractor** on a real galaxy cluster image
2. Visualize the detections overlaid on the coadd
3. Explore the measurements that Source Extractor produces

The image we are working with is the **Bullet Cluster** (1E 0657), one of the most famous galaxy clusters in the sky — it is the result of two clusters colliding at enormous speed, and it provided some of the strongest evidence for the existence of **dark matter**.

---

## Step 0 — Get the latest code

Before anything else, make sure you have the latest version of the `superbit-lensing` codebase. Open a terminal and run:

```bash
cd <you_path>/superbit-lensing
git pull
```

Then come back here and continue.

---

## Step 1 — Run Source Extractor

Source Extractor (or `sex`) is a program that scans an astronomical image, finds all the bright clumps of pixels that look like real objects (stars, galaxies, ...), and measures their properties.

We have wrapped it in a Python function so you can call it directly from this notebook.

In [ ]:
from superbit_lensing.utils import run_sextractor_coadd

coadd_file = "/projects/mccleary_group/ysp2026/se_coadd/1E0657_Bullet_coadd_b.fits"
cat_dir    = "/projects/mccleary_group/ysp2026/se_coadd/test_output"

In [ ]:
cat_file = run_sextractor_coadd(coadd_file, cat_dir=cat_dir)
print(f"Catalog saved to: {cat_file}")

---

## Step 2 — Load the image and the catalog

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from astropy.visualization import ZScaleInterval

# Load the coadd image (science data is in extension 0)
with fits.open(coadd_file) as hdul:
    image = hdul[0].data

print(f"Image shape: {image.shape}")

In [ ]:
catalog_file = "/projects/mccleary_group/ysp2026/se_coadd/test_output/1E0657_Bullet_coadd_b_cat.fits"

# Load the catalog as an astropy Table — easy to work with like a spreadsheet
cat = Table.read(catalog_file)

print(f"Number of detections: {len(cat)}")
print(f"Columns available:\n{cat.colnames}")

---

## Step 3 — Visualize detections on the image

We will use a **ZScale** stretch — the same one DS9 uses — to display the image, and draw a red circle around each detected source.

In [ ]:
# Pixel coordinates of each detection (SExtractor uses 1-based indexing)
x = cat['X_IMAGE']
y = cat['Y_IMAGE']

zscale = ZScaleInterval()
vmin, vmax = zscale.get_limits(image)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(image, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
ax.scatter(x - 1, y - 1, s=20, edgecolors='red', facecolors='none', linewidths=0.6)
ax.set_title("1E0657 Bullet Cluster — SExtractor detections", fontsize=14)
ax.set_xlabel("x [pixels]")
ax.set_ylabel("y [pixels]")
plt.tight_layout()
plt.show()

---

## Step 4 — Explore the catalog

Source Extractor did not just *find* sources — it also *measured* them. Each row in `cat` is one detected object, and each column is a different measurement.

Some useful columns to know about:

| Column | What it means |
|---|---|
| `X_IMAGE`, `Y_IMAGE` | Position in pixels |
| `MAG_AUTO` | Brightness (magnitude) — smaller = brighter |
| `T_ADMOM` | Size of the object (adaptive moments, in arcsec²) |
| `ELLIPTICITY` | How round vs. elongated the object is (0 = perfect circle) |
| `FLAGS` | Quality flags — 0 means a clean detection |

Let's take a quick peek at the first few rows:

In [ ]:
cat['X_IMAGE', 'Y_IMAGE', 'MAG_AUTO', 'T_ADMOM', 'ELLIPTICITY', 'FLAGS'][:10]

---

## Exercise

Now it is your turn! The catalog contains a lot of information about each object.

Try making a **scatter plot** with:
- `T_ADMOM` on the x-axis (size of the object)
- `MAG_AUTO` on the y-axis (brightness of the object)

What do you notice? Are larger objects brighter or fainter? Do you see any interesting structure in the plot?

> **Hint:** you can access a column like this: `cat['MAG_AUTO']`

In [ ]:
# Your code here!
fig, ax = plt.subplots(figsize=(8, 6))

# ax.scatter( ... )

ax.set_xlabel("T_ADMOM [arcsec²]")
ax.set_ylabel("MAG_AUTO")
ax.set_title("Size vs. Magnitude")
plt.tight_layout()
plt.show()